In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuredatabricks_catalog.ldp_medallion

In [0]:
folders = dbutils.fs.ls("abfss://raw@azuredatabricksansh.dfs.core.windows.net/rideshare")

for data_folder in folders:

    df = spark.read.format("csv").option("header", "true")\
            .option("inferSchema","true")\
            .load(data_folder.path)
    df = df.withColumnRenamed("updated_datetime",f"{data_folder.name.replace("/","")}_updated_datetime")
    
    if data_folder.name != 'trips/':
        df.write.format("delta")\
                .mode("overwrite")\
                .option("overwriteSchema","true")\
                .saveAsTable(f"azuredatabricks_catalog.ldp_medallion.{data_folder.name.replace("/","")}")

In [0]:
%sql
SELECT * FROM azuredatabricks_catalog.ldp_medallion.customers